In [4]:
# ── GIN v3: GINEConv + learned atom embeddings + richer features ──
import numpy as np
import pandas as pd
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GINEConv, global_add_pool, global_mean_pool

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm import tqdm
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ══════════════════════════════════════════════════════════════════════════════
# 1. Load data
# ══════════════════════════════════════════════════════════════════════════════
DATA_DIR = "/esneble_ai/tasks-2026/task1"

df_train = pd.read_parquet(f"{DATA_DIR}/chebi_dataset_train.parquet")
df_test = pd.read_parquet(f"{DATA_DIR}/chebi_dataset_test_empty.parquet")

label_cols = [c for c in df_train.columns if c.startswith("class_")]
num_classes = len(label_cols)

print(f"Train: {len(df_train)} molecules, {num_classes} classes")
print(f"Test:  {len(df_test)} molecules")

Using device: cpu


FileNotFoundError: [Errno 2] No such file or directory: '/esneble_ai/tasks-2026/task1/chebi_dataset_train.parquet'

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. Parse ChEBI hierarchy
# ══════════════════════════════════════════════════════════════════════════════
def parse_obo(path):
    child_to_parents = defaultdict(list)
    current_id = None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith("id: class_"):
                current_id = line.split("id: ")[1]
            elif line.startswith("is_a: class_"):
                parent = line.split("is_a: ")[1].strip()
                child_to_parents[current_id].append(parent)
    return child_to_parents

child_to_parents = parse_obo(f"{DATA_DIR}/chebi_classes.obo")

class_defs = pd.read_csv(f"{DATA_DIR}/chebi_class_definitions.csv")
class_id_to_name = dict(zip(class_defs["chebi_id"], class_defs["name"]))
class_to_idx = {c: i for i, c in enumerate(label_cols)}

parent_mask = torch.zeros(num_classes, num_classes)
for child, parents in child_to_parents.items():
    if child in class_to_idx:
        for p in parents:
            if p in class_to_idx:
                parent_mask[class_to_idx[child], class_to_idx[p]] = 1.0

parent_mask_np = parent_mask.numpy()
print(f"Hierarchy: {int(parent_mask.sum())} direct parent-child edges")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. Improved SMILES → PyG graph
#    - Categorical atom features (for learned embeddings): atomic_num, degree,
#      formal_charge, num_hs, hybridization
#    - Continuous atom features: is_aromatic, is_in_ring, mass, num_radical_e
#    - Bond features used via GINEConv (no longer ignored!)
# ══════════════════════════════════════════════════════════════════════════════

HYBRIDIZATION_MAP = {
    Chem.rdchem.HybridizationType.SP: 0,
    Chem.rdchem.HybridizationType.SP2: 1,
    Chem.rdchem.HybridizationType.SP3: 2,
    Chem.rdchem.HybridizationType.SP3D: 3,
    Chem.rdchem.HybridizationType.SP3D2: 4,
}

BOND_TYPE_MAP = {
    Chem.rdchem.BondType.SINGLE: 0,
    Chem.rdchem.BondType.DOUBLE: 1,
    Chem.rdchem.BondType.TRIPLE: 2,
    Chem.rdchem.BondType.AROMATIC: 3,
}

# Categorical feature vocab sizes (for nn.Embedding)
NUM_ATOMIC_NUM = 119   # 1..118 + unknown
NUM_DEGREE = 7         # 0..5 + unknown
NUM_FORMAL_CHARGE = 6  # -2..2 + unknown
NUM_NUM_HS = 6         # 0..4 + unknown
NUM_HYBRIDIZATION = 6  # 5 types + unknown
NUM_BOND_TYPE = 5      # 4 types + unknown

# Number of continuous atom features
NUM_CONT_ATOM_FEATS = 4  # is_aromatic, is_in_ring, normalized_mass, num_radical_e
# Number of continuous bond features
NUM_CONT_BOND_FEATS = 3  # is_conjugated, is_in_ring, stereo


def get_atom_features(atom):
    """Return (categorical_ids, continuous_features) for one atom."""
    atomic_num = atom.GetAtomicNum()
    cat = [
        min(atomic_num, NUM_ATOMIC_NUM - 1),            # atomic number (0-indexed, cap at 118)
        min(atom.GetDegree(), NUM_DEGREE - 2) + 1 if atom.GetDegree() <= 5 else NUM_DEGREE - 1,
        atom.GetFormalCharge() + 2 if -2 <= atom.GetFormalCharge() <= 2 else NUM_FORMAL_CHARGE - 1,
        min(atom.GetTotalNumHs(), NUM_NUM_HS - 2) + 1 if atom.GetTotalNumHs() <= 4 else NUM_NUM_HS - 1,
        HYBRIDIZATION_MAP.get(atom.GetHybridization(), NUM_HYBRIDIZATION - 1),
    ]
    cont = [
        1.0 if atom.GetIsAromatic() else 0.0,
        1.0 if atom.IsInRing() else 0.0,
        atom.GetMass() / 200.0,  # normalize mass roughly
        float(atom.GetNumRadicalElectrons()),
    ]
    return cat, cont


def get_bond_features(bond):
    """Return (categorical_id, continuous_features) for one bond."""
    bt = BOND_TYPE_MAP.get(bond.GetBondType(), NUM_BOND_TYPE - 1)
    cont = [
        1.0 if bond.GetIsConjugated() else 0.0,
        1.0 if bond.IsInRing() else 0.0,
        float(bond.GetStereo()),  # stereo info as numeric
    ]
    return bt, cont


def smiles_to_graph(smiles, labels=None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    atom_cats = []
    atom_conts = []
    for atom in mol.GetAtoms():
        c, f = get_atom_features(atom)
        atom_cats.append(c)
        atom_conts.append(f)

    x_cat = torch.tensor(atom_cats, dtype=torch.long)    # [N, 5]
    x_cont = torch.tensor(atom_conts, dtype=torch.float) # [N, 4]

    edge_index = []
    edge_cat = []
    edge_cont = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt, bf = get_bond_features(bond)
        edge_index.extend([[i, j], [j, i]])
        edge_cat.extend([bt, bt])
        edge_cont.extend([bf, bf])

    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_cat = torch.zeros((0,), dtype=torch.long)
        edge_cont = torch.zeros((0, NUM_CONT_BOND_FEATS), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_cat = torch.tensor(edge_cat, dtype=torch.long)
        edge_cont = torch.tensor(edge_cont, dtype=torch.float)

    data = Data(
        x_cat=x_cat,
        x_cont=x_cont,
        edge_index=edge_index,
        edge_cat=edge_cat,
        edge_cont=edge_cont,
    )
    if labels is not None:
        data.y = torch.tensor(labels, dtype=torch.float)
    return data


sample = smiles_to_graph("CCO")
print(f"Atom categorical features: {sample.x_cat.shape}")
print(f"Atom continuous features:  {sample.x_cont.shape}")
print(f"Edge categorical features: {sample.edge_cat.shape}")
print(f"Edge continuous features:  {sample.edge_cont.shape}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. Convert all molecules
# ══════════════════════════════════════════════════════════════════════════════
def build_graph_dataset(df, label_cols=None):
    graphs = []
    skipped = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Building graphs"):
        labels = row[label_cols].values.astype(np.float32) if label_cols else None
        g = smiles_to_graph(row["SMILES"], labels)
        if g is not None:
            g.mol_id = row["mol_id"]
            graphs.append(g)
        else:
            skipped += 1
    print(f"Built {len(graphs)} graphs, skipped {skipped}")
    return graphs

train_graphs = build_graph_dataset(df_train, label_cols)
test_graphs = build_graph_dataset(df_test, label_cols=None)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5. Train/val split & dataloaders
# ══════════════════════════════════════════════════════════════════════════════
train_data, val_data = train_test_split(train_graphs, test_size=0.1, random_state=42)

BATCH_SIZE = 128
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_graphs)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 6. GIN v3 Model — GINEConv + learned embeddings
# ══════════════════════════════════════════════════════════════════════════════

class GINEBlock(nn.Module):
    def __init__(self, hidden_dim, edge_dim):
        super().__init__()
        mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.conv = GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
        self.bn = nn.BatchNorm1d(hidden_dim)

    def forward(self, x, edge_index, edge_attr):
        x = self.conv(x, edge_index, edge_attr)
        x = self.bn(x)
        return F.relu(x)


class ImprovedGIN(nn.Module):
    def __init__(self, hidden_dim, num_classes, num_layers=5, dropout=0.3,
                 atom_emb_dim=64, bond_emb_dim=32):
        super().__init__()
        self.num_layers = num_layers
        self.dropout = dropout

        # Learned embeddings for categorical atom features
        self.atom_emb_atomic_num = nn.Embedding(NUM_ATOMIC_NUM, atom_emb_dim)
        self.atom_emb_degree = nn.Embedding(NUM_DEGREE, 16)
        self.atom_emb_formal_charge = nn.Embedding(NUM_FORMAL_CHARGE, 16)
        self.atom_emb_num_hs = nn.Embedding(NUM_NUM_HS, 16)
        self.atom_emb_hybridization = nn.Embedding(NUM_HYBRIDIZATION, 16)

        # Total atom input dim: 64 + 16*4 + 4 continuous = 132
        atom_input_dim = atom_emb_dim + 16 * 4 + NUM_CONT_ATOM_FEATS
        self.input_proj = nn.Linear(atom_input_dim, hidden_dim)

        # Learned embedding for categorical bond features
        self.bond_emb = nn.Embedding(NUM_BOND_TYPE, bond_emb_dim)
        # Total edge dim: 32 + 3 continuous = 35
        edge_dim = bond_emb_dim + NUM_CONT_BOND_FEATS
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)

        # GINEConv layers (edge-aware!)
        self.gin_layers = nn.ModuleList()
        for _ in range(num_layers):
            self.gin_layers.append(GINEBlock(hidden_dim, hidden_dim))

        # Jumping Knowledge
        self.jk_proj = nn.Linear(hidden_dim * (num_layers + 1), hidden_dim)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def encode_atoms(self, x_cat, x_cont):
        """Combine learned embeddings with continuous features."""
        emb = torch.cat([
            self.atom_emb_atomic_num(x_cat[:, 0]),
            self.atom_emb_degree(x_cat[:, 1]),
            self.atom_emb_formal_charge(x_cat[:, 2]),
            self.atom_emb_num_hs(x_cat[:, 3]),
            self.atom_emb_hybridization(x_cat[:, 4]),
            x_cont,
        ], dim=-1)
        return emb

    def encode_edges(self, edge_cat, edge_cont):
        """Combine learned bond embedding with continuous features."""
        emb = torch.cat([
            self.bond_emb(edge_cat),
            edge_cont,
        ], dim=-1)
        return self.edge_proj(emb)

    def forward(self, data):
        x_cat, x_cont = data.x_cat, data.x_cont
        edge_index, batch = data.edge_index, data.batch
        edge_cat, edge_cont = data.edge_cat, data.edge_cont

        # Encode atoms and edges
        x = self.encode_atoms(x_cat, x_cont)
        x = F.relu(self.input_proj(x))
        x = F.dropout(x, p=self.dropout, training=self.training)

        edge_attr = self.encode_edges(edge_cat, edge_cont)

        # Message passing with edge features
        layer_outputs = [x]
        for gin in self.gin_layers:
            x = gin(x, edge_index, edge_attr)
            x = F.dropout(x, p=self.dropout, training=self.training)
            layer_outputs.append(x)

        # Jumping knowledge
        x = torch.cat(layer_outputs, dim=-1)
        x = self.jk_proj(x)

        # Readout
        x_add = global_add_pool(x, batch)
        x_mean = global_mean_pool(x, batch)
        x = torch.cat([x_add, x_mean], dim=-1)

        return self.classifier(x)


HIDDEN_DIM = 256
NUM_LAYERS = 5
DROPOUT = 0.3

model = ImprovedGIN(
    hidden_dim=HIDDEN_DIM,
    num_classes=num_classes,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7. Loss — BCE with pos_weight capped at 20
# ══════════════════════════════════════════════════════════════════════════════
all_labels = np.array([g.y.numpy() for g in train_graphs])
pos_freq = all_labels.mean(axis=0)
pos_freq = np.clip(pos_freq, 0.001, 0.999)
pos_weight = torch.tensor((1 - pos_freq) / pos_freq, dtype=torch.float).to(device)
pos_weight = torch.clamp(pos_weight, max=20.0)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"Pos weight range: [{pos_weight.min():.2f}, {pos_weight.max():.2f}]")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 8. Evaluation
# ══════════════════════════════════════════════════════════════════════════════
def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch)
            probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())
            labels = batch.y.view(probs.shape).cpu().numpy()
            all_labels.append(labels)
    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    all_preds = (all_probs >= threshold).astype(float)

    f1_micro = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    # Quick binary violation count
    violations = 0
    total_pairs = 0
    for child_idx in range(parent_mask_np.shape[0]):
        parent_indices = np.where(parent_mask_np[child_idx] > 0)[0]
        for p_idx in parent_indices:
            violations += ((all_preds[:, child_idx] == 1) & (all_preds[:, p_idx] == 0)).sum()
            total_pairs += len(all_preds)

    return f1_micro, f1_macro, int(violations), total_pairs, all_probs, all_preds, all_labels

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 9. Training — LR 5e-4 + cosine annealing
# ══════════════════════════════════════════════════════════════════════════════
LR = 5e-4
EPOCHS = 60
PATIENCE = 10

optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=2, eta_min=1e-6)

best_f1 = 0
patience_counter = 0
history = {"train_loss": [], "val_f1_micro": [], "val_f1_macro": [], "violations": [], "viol_rate": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch)
        loss = criterion(logits, batch.y.view(logits.shape))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs

    scheduler.step()
    avg_loss = total_loss / len(train_data)

    f1_micro, f1_macro, violations, total_pairs, _, _, _ = evaluate(model, val_loader)

    viol_rate = violations / total_pairs * 100 if total_pairs > 0 else 0
    history["train_loss"].append(avg_loss)
    history["val_f1_micro"].append(f1_micro)
    history["val_f1_macro"].append(f1_macro)
    history["violations"].append(violations)
    history["viol_rate"].append(viol_rate)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch:02d} | Loss: {avg_loss:.4f} | "
          f"F1-micro: {f1_micro:.4f} | F1-macro: {f1_macro:.4f} | "
          f"Violations: {violations:,} ({viol_rate:.2f}%) | LR: {current_lr:.2e}")

    if f1_micro > best_f1:
        best_f1 = f1_micro
        patience_counter = 0
        torch.save(model.state_dict(), "best_model_gin3.pt")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest validation F1-micro: {best_f1:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 10. Training curves
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["train_loss"])
axes[0].set_title("Train Loss")
axes[0].set_xlabel("Epoch")

axes[1].plot(history["val_f1_micro"], label="F1-micro")
axes[1].plot(history["val_f1_macro"], label="F1-macro")
axes[1].set_title("Validation F1")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(history["viol_rate"], color="red")
axes[2].set_title("Violation Rate % (val)")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("%")

plt.tight_layout()
plt.savefig("training_curves_gin3.png", dpi=150)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 11. Violation analysis on validation set
# ══════════════════════════════════════════════════════════════════════════════
def count_violations_detailed(probs, preds, parent_mask_np, label_cols, class_id_to_name):
    n_samples = preds.shape[0]
    binary_violations = 0
    prob_violations = 0
    total_pairs = 0
    edge_violations = []
    per_sample_violations = np.zeros(n_samples)

    for child_idx in range(parent_mask_np.shape[0]):
        parent_indices = np.where(parent_mask_np[child_idx] > 0)[0]
        for p_idx in parent_indices:
            total_pairs += n_samples
            bin_v = ((preds[:, child_idx] == 1) & (preds[:, p_idx] == 0)).sum()
            binary_violations += bin_v
            prob_diff = probs[:, child_idx] - probs[:, p_idx]
            prob_v_mask = prob_diff > 0
            n_prob_v = prob_v_mask.sum()
            prob_violations += n_prob_v
            per_sample_violations += (preds[:, child_idx] > preds[:, p_idx]).astype(float)
            if bin_v > 0 or n_prob_v > 0:
                child_name = class_id_to_name.get(label_cols[child_idx], label_cols[child_idx])
                parent_name = class_id_to_name.get(label_cols[p_idx], label_cols[p_idx])
                mean_diff = prob_diff[prob_v_mask].mean() if n_prob_v > 0 else 0.0
                edge_violations.append((
                    label_cols[child_idx], child_name,
                    label_cols[p_idx], parent_name,
                    int(bin_v), int(n_prob_v), float(mean_diff)
                ))

    return {
        "binary_violations": int(binary_violations),
        "prob_violations": int(prob_violations),
        "total_pairs": total_pairs,
        "edge_violations": sorted(edge_violations, key=lambda x: -x[4]),
        "per_sample_violations": per_sample_violations,
    }


print("Loading best model for violation analysis...")
model.load_state_dict(torch.load("best_model_gin3.pt", weights_only=True))

f1_micro, f1_macro, violations, total_pairs, val_probs, val_preds, val_labels = evaluate(model, val_loader)
print(f"Best model — F1-micro: {f1_micro:.4f}, F1-macro: {f1_macro:.4f}")
print(f"Violations: {violations:,} / {total_pairs:,} ({violations/total_pairs*100:.2f}%)")

stats = count_violations_detailed(val_probs, val_preds, parent_mask_np, label_cols, class_id_to_name)

print(f"\nSamples with violations: {(stats['per_sample_violations'] > 0).sum()} / {len(stats['per_sample_violations'])}")
print(f"Mean violations per sample: {stats['per_sample_violations'].mean():.2f}")
print(f"\nTop 15 most violated edges:")
print(f"{'Child':<35} {'Parent':<35} {'Bin':>6} {'Prob':>6}")
print("-" * 85)
for edge in stats["edge_violations"][:15]:
    child_id, child_name, parent_id, parent_name, bin_v, prob_v, mean_d = edge
    print(f"{child_name[:32]:<35} {parent_name[:32]:<35} {bin_v:>6} {prob_v:>6}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 12. Generate test predictions & submission
# ══════════════════════════════════════════════════════════════════════════════
model.eval()
all_preds = []
all_mol_ids = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        logits = model(batch)
        preds = (torch.sigmoid(logits) >= 0.5).int()
        all_preds.append(preds.cpu().numpy())
        all_mol_ids.extend(batch.mol_id)

all_preds = np.vstack(all_preds)

submission = pd.DataFrame(all_preds, columns=label_cols)
submission.insert(0, "mol_id", all_mol_ids)
submission = df_test[["mol_id", "SMILES"]].merge(submission, on="mol_id", how="left")
submission[label_cols] = submission[label_cols].fillna(0).astype(int)

# Count violations in submission
pred_vals = submission[label_cols].values
violations = 0
for child_idx in range(parent_mask_np.shape[0]):
    parent_indices = np.where(parent_mask_np[child_idx] > 0)[0]
    for p_idx in parent_indices:
        violations += ((pred_vals[:, child_idx] == 1) & (pred_vals[:, p_idx] == 0)).sum()

print(f"Submission shape: {submission.shape}")
print(f"Hierarchy violations in test submission: {violations:,}")

submission.to_parquet("chebi_submission_gin3.parquet", index=False)
print("Saved to chebi_submission_gin3.parquet")